# 04.4 Process Optimization: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Object creation, method calls, keyword arguments and lists.

**Practises:** process variables, linear and purity constraints, objective direction, nested simulation/fractionation evaluation and verification of the selected process.

1. Define the problem for a single product
2. Run and verify the search
3. Relax the purity requirement (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

In the lesson, both **A and B** were products. Here, only **A** is the product: B is collected as waste. Adapt the optimization problem to this specification, then compare the selected operating point with the baseline. Exercises marked **optional** extend the main tasks and can be skipped.

## Starter process

The supplied process is the batch elution from the lesson: a **60 cm × 2.4 cm** column at **60 mL/min**, feeding A and B (10 mol/m³ each) for **60 s** in a **10 min** cycle. The simulator runs to cyclic stationarity, and the fractionation optimizer uses the purity tolerance from the lesson.

```{tip}
Run the cell below once with **Shift+Enter** to load the process, simulator and fractionation optimizer. The [setup script](../resources/process_optimization_setup.py) contains the model.
```

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
# Load the batch-elution model and the comparison plot from the lesson.
%run ../resources/process_optimization_setup.py
from CADETProcess.simulator import Cadet
from CADETProcess.fractionation import FractionationOptimizer
from CADETProcess.optimization import OptimizationProblem, U_NSGA3

process = batch_elution_process()
process_simulator = Cadet()
process_simulator.evaluate_stationarity = True
fractionation_optimizer = FractionationOptimizer()
fractionation_optimizer.optimizer.catol = 1e-4  # Absolute purity-constraint tolerance.

## Exercise 1: define the problem for product A

Create an `OptimizationProblem` named `optimization_problem` for `process`:

1. Add the variables **in this order**, with `normalization="linear"`:

   | Variable name | Parameter path | Lower bound | Upper bound |
   |---|---|---:|---:|
   | `cycle_time` | `cycle_time` | 8 min | 15 min |
   | `feed_duration` | `feed_duration.time` | 20 s | 90 s |

2. Keep at least **2 min** between the end of feeding and the next injection: **feed duration − cycle time ≤ −120 s**.
3. Register the simulator, then the fractionation optimizer with `purity_required=[0.951, 0]`: only A is collected as a product. The small margin above 95% keeps the achieved purity above the constraint despite the solver tolerance, as recommended in Fractionation.
4. Maximize the productivity of **A only**, and require a purity of **at least 95% for A**. Both need the simulation and fractionation results.

Times are in **seconds**: convert minutes with `* 60`.

```{tip}
**Methods for this exercise** (as in the lesson)

- `add_variable(name, targets=[process], parameter_path=..., lb=..., ub=..., normalization="linear")` and `add_linear_constraint([names], lhs=[...], b=...)`. See the [optimization problem guide](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/optimization_problem.html).
- `add_evaluator(fractionation_optimizer, kwargs={"purity_required": [...]})` passes settings to the fractionation. See the [evaluator guide](https://cadet-process.readthedocs.io/en/latest/user_guide/optimization/evaluator.html).
- `Productivity(ranking=[1, 0])` weights only A. Use `add_objective(..., minimize=False, requires=[process_simulator, fractionation_optimizer])`. See [`Productivity`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.performance.Productivity.html).
- `Purity()` returns one value per component. Use `add_nonlinear_constraint(purity, n_nonlinear_constraints=2, bounds=[...], comparison_operator="ge", requires=[...])`; a bound of `0` leaves B unconstrained. See [`Purity`](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.performance.Purity.html).
```

In [ ]:
# Optimization problem for product A
from CADETProcess.performance import Productivity, Purity

optimization_problem = OptimizationProblem("batch_elution_A")
optimization_problem.add_evaluation_object(process)

optimization_problem.add_variable(
    "cycle_time", targets=[process], parameter_path="cycle_time",
    lb=8 * 60, ub=15 * 60, normalization="linear",
)
optimization_problem.add_variable(
    "feed_duration", targets=[process], parameter_path="feed_duration.time",
    lb=20, ub=90, normalization="linear",
)
optimization_problem.add_linear_constraint(
    ["feed_duration", "cycle_time"], lhs=[1, -1], b=-2 * 60,
)

optimization_problem.add_evaluator(process_simulator)
optimization_problem.add_evaluator(
    fractionation_optimizer, kwargs={"purity_required": [0.951, 0]},
)

optimization_problem.add_objective(
    Productivity(ranking=[1, 0]), name="productivity_A", minimize=False,
    requires=[process_simulator, fractionation_optimizer],
)
optimization_problem.add_nonlinear_constraint(
    Purity(), name="purity", n_nonlinear_constraints=2,
    bounds=[0.95, 0], comparison_operator="ge",
    requires=[process_simulator, fractionation_optimizer],
)

Check one candidate before the search. The baseline **10 min cycle / 60 s feed** should give a positive productivity and an A purity just above **0.95**.

In [ ]:
# Evaluate the baseline [cycle time, feed duration] through the complete problem definition.
baseline = [10 * 60, 60]
print("Baseline productivity of A:", optimization_problem.evaluate_objectives(baseline))
print("Baseline purities [A, B]:", optimization_problem.evaluate_nonlinear_constraints(baseline))

```{dropdown} Interpretation
A positive productivity confirms the objective direction and the parameter paths. The A purity is just above 0.95 because the fractionation optimizer widens A's collection window until it reaches the requested 0.951. B's purity is reported, but its bound of 0 never restricts a candidate.
```

## Exercise 2: run and verify the search

The supplied cells run the same short search as the lesson (**8 candidates, 3 generations**, the baseline in the starting population), then apply the selected values and compare them with the baseline. Before running: do you expect a longer or shorter feed than in the lesson, now that B no longer has to be pure?

In [ ]:
# Run a short search; the first starting candidate is the checked baseline.
optimizer = U_NSGA3()
optimizer.n_cores = 1
optimizer.pop_size = 8
optimizer.n_max_iter = 3
optimizer.seed = 42
initial_population = optimization_problem.create_initial_values(
    optimizer.pop_size, seed=optimizer.seed,
)
initial_population[0] = baseline
results = optimizer.optimize(optimization_problem, x0=initial_population)
print("Selected [cycle time, feed duration] / s:", results.x)

In [ ]:
# Simulate baseline and selected settings, then compare settings and product pools.
baseline_process = batch_elution_process()
baseline_fractionation = fractionation_optimizer.optimize_fractionation(
    process_simulator.simulate(baseline_process), purity_required=[0.951, 0],
)

process.cycle_time = float(results.x[0, 0])
process.feed_duration.time = float(results.x[0, 1])
selected_fractionation = fractionation_optimizer.optimize_fractionation(
    process_simulator.simulate(process), purity_required=[0.951, 0],
)
print(selected_fractionation.performance)
_ = plot_process_comparison(
    baseline, results.x[0], baseline_fractionation, selected_fractionation,
)

```{dropdown} Interpretation
Only A's pool has to be pure, so B's peak may overlap with A's tail as long as A's collection window still reaches 95% purity. With the supplied seed, the search shortens the cycle to about **8 min**, close to its lower bound, and lengthens the feed slightly; productivity of A rises by about a quarter at the required purity. A variable at its bound suggests that a wider range might allow further improvement. Check A's purity and recovery alongside its productivity; this short search does not establish a global optimum.
```

## Exercise 3 (optional): relax the purity requirement

Repeat the problem with a **90%** purity requirement for A. Copy your Exercise 1 cell into the answer cell, name the problem `relaxed_problem`, and change the requirement in **both** places: the fractionation optimizer's `purity_required` (with margin, `0.901`) and the purity constraint's `bounds` (`0.90`).

```{warning}
The copied cell must register the same `process`: the supplied cells below reset it to the baseline first, so both searches start from the same settings.
```

In [ ]:
# Reset the process to the baseline before defining and running the relaxed problem.
process.cycle_time = 10 * 60
process.feed_duration.time = 60

In [ ]:
# Relaxed problem with 90% purity
relaxed_problem = OptimizationProblem("batch_elution_A_relaxed")
relaxed_problem.add_evaluation_object(process)

relaxed_problem.add_variable(
    "cycle_time", targets=[process], parameter_path="cycle_time",
    lb=8 * 60, ub=15 * 60, normalization="linear",
)
relaxed_problem.add_variable(
    "feed_duration", targets=[process], parameter_path="feed_duration.time",
    lb=20, ub=90, normalization="linear",
)
relaxed_problem.add_linear_constraint(
    ["feed_duration", "cycle_time"], lhs=[1, -1], b=-2 * 60,
)

relaxed_problem.add_evaluator(process_simulator)
relaxed_problem.add_evaluator(
    fractionation_optimizer, kwargs={"purity_required": [0.901, 0]},
)

relaxed_problem.add_objective(
    Productivity(ranking=[1, 0]), name="productivity_A", minimize=False,
    requires=[process_simulator, fractionation_optimizer],
)
relaxed_problem.add_nonlinear_constraint(
    Purity(), name="purity", n_nonlinear_constraints=2,
    bounds=[0.90, 0], comparison_operator="ge",
    requires=[process_simulator, fractionation_optimizer],
)

In [ ]:
# Run the same short search for the relaxed requirement and compare productivities.
relaxed_population = relaxed_problem.create_initial_values(optimizer.pop_size, seed=optimizer.seed)
relaxed_population[0] = baseline
relaxed_results = optimizer.optimize(relaxed_problem, x0=relaxed_population)
print("Selected [cycle time, feed duration] / s at 95%:", results.x[0])
print("Selected [cycle time, feed duration] / s at 90%:", relaxed_results.x[0])
print("Productivity of A at 95%:", results.f[0])
print("Productivity of A at 90%:", relaxed_results.f[0])

```{dropdown} Interpretation
A lower purity requirement accepts more of the overlap between A and B in the product pool. With the supplied seed, the feed grows to almost **90 s**, its upper bound, and productivity of A rises by about 40% compared with Exercise 2. The specification therefore shapes the optimal process as much as the model does: agree on it before optimizing.
```